# This notebook will serve as a way to implement character generation RNN and other implementation (trained on Kaggle)

In [1]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.nn as nn
import numpy as np
import pandas as pd

In [2]:
with open("..\Corpus\clean_corpus_Medine.txt", "r", encoding="utf-8", errors="replace") as f:
    corpus = f.read()

# Data preparation

## We need first to create a mapping of each character for encoding

In [3]:
print(f"Length of the corpus characters : {len(corpus)}")

Length of the corpus characters : 571773


In [4]:
chars_sort = sorted(set(corpus))

#Encode
char2int = {ch: i for i, ch in enumerate(chars_sort)}
print(char2int,"\n")

# Decode
int2char = {i: ch for ch, i in char2int.items()}
print(int2char)

nb_char = len(int2char)

{'\n': 0, ' ': 1, '!': 2, '%': 3, '&': 4, "'": 5, ')': 6, '+': 7, ',': 8, '-': 9, '.': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, ':': 22, ';': 23, '<': 24, '>': 25, '?': 26, 'A': 27, 'B': 28, 'C': 29, 'D': 30, 'E': 31, 'F': 32, 'G': 33, 'H': 34, 'I': 35, 'J': 36, 'K': 37, 'L': 38, 'M': 39, 'N': 40, 'O': 41, 'P': 42, 'Q': 43, 'R': 44, 'S': 45, 'T': 46, 'U': 47, 'V': 48, 'W': 49, 'X': 50, 'Y': 51, 'Z': 52, 'a': 53, 'b': 54, 'c': 55, 'd': 56, 'e': 57, 'f': 58, 'g': 59, 'h': 60, 'i': 61, 'j': 62, 'k': 63, 'l': 64, 'm': 65, 'n': 66, 'o': 67, 'p': 68, 'q': 69, 'r': 70, 's': 71, 't': 72, 'u': 73, 'v': 74, 'w': 75, 'x': 76, 'y': 77, 'z': 78, '§': 79, 'À': 80, 'Ç': 81, 'È': 82, 'É': 83, 'Ê': 84, 'Ô': 85, 'Ö': 86, 'à': 87, 'á': 88, 'â': 89, 'ã': 90, 'æ': 91, 'ç': 92, 'è': 93, 'é': 94, 'ê': 95, 'ë': 96, 'í': 97, 'î': 98, 'ï': 99, 'ñ': 100, 'ô': 101, 'ö': 102, 'ù': 103, 'û': 104, 'ü': 105, 'Ā': 106, 'ā': 107, 'ğ': 108, 'ō': 109, 'Œ': 110

Encode the corpus

In [5]:
corpus_encoded = np.array([char2int[ch] for ch in corpus])
corpus_encoded

array([24, 29, 41, ...,  0, 79,  0])

## Creation of the dataset

To increase the randomness during the training :

For each epoch the entire corpus will have a random specific offset value in order that the model during training doesn't see the exact same text during X epochs.

In [6]:
class CharDataset(Dataset) :
    def __init__(self, text, length_seq) :
        self.text = text
        self.length_seq = length_seq
        self.max_start = len(text) - length_seq - 1
        self.offset = 0

        # ensure text is torch tensor
        if not torch.is_tensor(text):
            text = torch.tensor(text, dtype=torch.long)
        self.text = text
        
    def set_offset(self, offset) :
        self.offset = offset
        
    def __len__(self) :
        return self.max_start

    def __getitem__(self, i) :
        s = (i + self.offset) % self.max_start
        x = self.text[s:s+self.length_seq]
        y = self.text[s+1:s+self.length_seq+1]
        return x, y

In [7]:
train_size = int(len(corpus_encoded)*0.9)
train_encoded = torch.tensor(corpus_encoded[:train_size], dtype=torch.long)
test_encoded = torch.tensor(corpus_encoded[train_size:], dtype=torch.long)

In [8]:
seq_length = 250
train_dataset = CharDataset(train_encoded, length_seq= seq_length)
test_dataset = CharDataset(test_encoded, length_seq= seq_length)

In [9]:
print(train_dataset[0][0])
print("".join([int2char[i] for i in train_encoded[:seq_length].numpy()]))

tensor([24, 29, 41, 47, 42, 38, 31, 46, 25,  0, 38, 53,  1, 54, 53, 66, 64, 61,
        57, 73, 57,  8,  1, 55,  5, 57, 71, 72,  1, 56, 57, 71,  1, 68, 89, 69,
        73, 57, 70, 57, 72, 72, 57, 71,  1, 71, 73, 70,  1, 73, 66,  1, 72, 53,
        71,  1, 56,  5, 58, 73, 65, 61, 57, 70,  1,  0, 38, 57, 71,  1, 74, 67,
        61, 76,  1, 56, 73,  1, 59, 60, 57, 72, 72, 67,  1, 71, 67, 66, 72,  1,
        53, 73, 72, 67,  9, 72, 73, 66, 94, 57, 71,  1,  0, 79,  0, 24, 29, 41,
        47, 42, 38, 31, 46, 25,  0,  1,  0, 42, 53, 68, 77,  1, 72,  5, 53,  1,
        56, 94, 58, 57, 66, 56, 73,  8,  1, 57, 71, 72,  9, 55,  5, 69, 73, 57,
         1, 72, 73,  1, 72,  5, 57, 66,  1, 71, 67, 73, 74, 61, 57, 66, 71,  1,
        26,  1,  0, 79,  0, 24, 29, 41, 47, 42, 38, 31, 46, 25,  0, 68, 67, 73,
        70,  1, 69, 73,  5, 64, 53,  1, 70, 53, 58, 64, 57, 73, 71, 57,  1, 72,
        57,  1, 72, 67, 73, 55, 60, 57,  1,  0, 79,  0, 24, 44, 31, 32, 44, 27,
        35, 40, 25,  0, 29,  5, 57, 71, 

## Creation of Dataloader and co

200 context for previous model

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
batch_size = 256 #Trained on 256 batch_size

if device == "cpu" :

    train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=True) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

else : 
    train_dl = DataLoader(train_dataset, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                        num_workers=4, prefetch_factor=4, shuffle=False, drop_last=True) #Shuffle False because we need the RNN to use previous sequences data to predict next one
    test_dl = DataLoader(test_dataset, batch_size=batch_size, pin_memory=True, pin_memory_device=device, 
                       num_workers=2, prefetch_factor=2, shuffle=False, drop_last=True)

### Check the good dataloading and offset validity

In [12]:
for i in train_dl :
    print(i[0])
    break

tensor([[24, 29, 41,  ...,  1,  0, 29],
        [29, 41, 47,  ...,  0, 29,  5],
        [41, 47, 42,  ..., 29,  5, 57],
        ...,
        [72,  1, 66,  ..., 61,  1, 69],
        [ 1, 66, 67,  ...,  1, 69, 73],
        [66, 67, 73,  ..., 69, 73, 53]])


In [13]:
train_dataset.set_offset(1)
next(iter(train_dl))

[tensor([[29, 41, 47,  ...,  0, 29,  5],
         [41, 47, 42,  ..., 29,  5, 57],
         [47, 42, 38,  ...,  5, 57, 71],
         ...,
         [ 1, 66, 67,  ...,  1, 69, 73],
         [66, 67, 73,  ..., 69, 73, 53],
         [67, 73, 71,  ..., 73, 53, 66]]),
 tensor([[41, 47, 42,  ..., 29,  5, 57],
         [47, 42, 38,  ...,  5, 57, 71],
         [42, 38, 31,  ..., 57, 71, 72],
         ...,
         [66, 67, 73,  ..., 69, 73, 53],
         [67, 73, 71,  ..., 73, 53, 66],
         [73, 71,  8,  ..., 53, 66, 56]])]

Seems to work

In [14]:
len(train_dl)

2009

## Models

In [15]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, num_layers=1):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True, dropout = 0.15, nonlinearity ="relu")
        self.drop = nn.Dropout(p=0.15)
        self.ln = nn.LayerNorm(hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embedding(x)              # (batch, seq, hidden_size)
        out, hidden = self.rnn(x, hidden)
        out = self.ln(out)
        out = self.drop(out)
        out = self.fc(out)                  
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size)

### Training part

In [ ]:
vocab_size = len(char2int)
hidden_size = 236
num_epoch = 50

nb_step_train = len(train_dl)
nb_step_test = len(test_dl)

model = CharRNN(vocab_size, hidden_size, num_layers=3)   

model.to(device)

loss_fn = nn.CrossEntropyLoss()

opti = torch.optim.AdamW(model.parameters(), lr=0.0015, weight_decay=1e-3)
sched_warm = torch.optim.lr_scheduler.LinearLR(opti, start_factor=0.2, end_factor=1.0, total_iters=500)
sched_post = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opti, T_0=len(train_dl)*10, T_mult=2, eta_min=0.0001) #1 epoch => 2 => 4 => 8

In [ ]:
list_offset = []
l_tot = []

#Early stopping
early_stopping_count = 0
patience = 5

best_val = float("inf")

scaler = torch.amp.GradScaler()

hid = model.init_hidden(batch_size).to(device)

hid_ = model.init_hidden(batch_size).to(device)

for epoch in range(num_epoch) :
    
    offset = np.random.randint(0, seq_length-1) #Set the offset
    list_offset.append(offset) #Keep in memory

    #Offset the datas
    train_dataset.set_offset(offset); test_dataset.set_offset(offset)

    model.train()
    hid.zero_()

    #Create loss per epoch
    l_train = 0.0
    l_test = 0.0
    
    for X,Y in iter(train_dl) :
        X = X.to(device); Y= Y.to(device)
        opti.zero_grad(set_to_none=True)
        
        #Computation of model
        hid = hid.detach();
        
        with torch.amp.autocast(device_type="cuda:0"):
            pred, hid = model(X, hid)
            loss = loss_fn(pred.view(-1, vocab_size), Y.view(-1))

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
        scaler.step(opti); scaler.update()
        l_train += loss.detach()

        #Scheduler part
        #Warm start
        
        if sched_post.T_cur == 0 and epoch != 0:  #After warm restart decrease the max learning rate
            sched_post.base_lrs[0] = sched_post.base_lrs[0] * 0.6
            sched_post.eta_min = sched_post.eta_min * 1.25
            print(f"Decrease {sched_post.base_lrs[0]}, {sched_post.eta_min}")

        step_scheduler = sched_warm if (epoch == 0 and sched_warm.last_epoch < 1500) else sched_post
        step_scheduler.step()

    #Test data part
    
    model.eval()
    
    
    with torch.inference_mode():
        hid_.zero_()  
        with torch.amp.autocast("cuda"):
            for X,Y in iter(test_dl) : 
                X = X.to(device); Y= Y.to(device)
                hid_ = hid_.detach()
                            
                pred, hid_ = model(X, hid_)
                loss = loss_fn(pred.view(-1, vocab_size), Y.view(-1))
                l_test += loss.detach()

        print(epoch,np.exp(l_test.item()/nb_step_test),"\n")
     
        #Record the loss of the epoch
        l_tot.append(l_test.item()); 

        if l_test < best_val :
            best_val = l_test.item()
            early_stopping_count = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": opti.state_dict(),
                "scheduler_state_dict": sched_post.state_dict(),
                "val_loss": l_test,
            }, "model1")
        
        elif l_test >= best_val :
            early_stopping_count += 1

        if early_stopping_count == patience :
            print("Early Stopping")
            break 

print(f"Liste of offset used : {list_offset}")

0 5.9641423395348685 

1 5.491976977073773 

2 5.317788120865117 

3 5.138765347598326 

4 5.011612804589071 

5 4.9137427846943345 

6 4.84087926343651 

7 4.787608116480671 

8 4.750376723626298 

9 4.727642397358877 

Decrease 0.0009, 0.000125
10 4.786922491723389 

11 4.746652836678526 

12 4.708421967699388 

13 4.674828882949981 

14 4.644281297093341 

15 4.615863635333385 

16 4.5919915073867825 

17 4.569703839533204 

18 4.550087010104633 

19 4.532226573526176 

20 4.516975424689546 

21 4.50161110125757 

22 4.488072626890405 

23 4.475963974359692 

24 4.464798653684763 

25 4.455590641346918 

26 4.447188472551699 

27 4.439601324126818 

28 4.433963821115854 

29 4.4299898688197485 

Decrease 0.00054, 0.00015625
30 4.461622297272576 

31 4.455712303808094 

32 4.449636821864406 

33 4.442766760456447 

34 4.437306976524796 

Early Stopping
Liste of offset used : [174, 151, 166, 224, 93, 247, 103, 150, 111, 188, 160, 229, 119, 240, 142, 127, 87, 139, 1, 65, 28, 34, 50, 14

In [ ]:
torch.save(model.state_dict(), "modele.pth")

### Post training

In [16]:
matrix = pd.read_csv(r"..\Markov\transition_matrix.csv")

In [17]:
states = np.array(matrix.iloc[6])
prob_transi = np.array(matrix.iloc[0:6])

In [18]:
vocab_size = len(char2int)
hidden_size = 236
num_epoch = 50

model = CharRNN(vocab_size, hidden_size, num_layers=3)   
model.to("cpu")

ckpt = torch.load("model_RNN", map_location="cpu")
model.load_state_dict(ckpt["model_state_dict"])

<All keys matched successfully>

In [19]:
def sample_with_temp_topk(logits, temperature=1.0, top_k=50):
    # appliquer température
    logits = logits / temperature
    
    # garder seulement les top_k logits
    if top_k is not None:
        values, indices = torch.topk(logits, top_k)
        mask = torch.full_like(logits, float('-inf'))
        mask[indices] = logits[indices]
        logits = mask
    
    # convertir en proba
    probs = torch.softmax(logits, dim=-1)
    
    # tirage
    return torch.multinomial(probs, 1)

In [20]:
def generate_a_song_structure(matrix,states) :
    
    song_struct = [0]

    index = [i for i, p in enumerate(matrix[0]) if p>0]
    proba = [p for p in matrix[0] if p > 0]

    cumsum = np.cumsum(proba)
    r = np.random.rand()

    idx = np.searchsorted(cumsum, r)
    selected_value = index[idx] 
    song_struct.append(selected_value)

    end = False

    while end == False :

        index = [i for i, p in enumerate(matrix[selected_value]) if p>0]
        proba = [p for p in matrix[selected_value] if p > 0]

        cumsum = np.cumsum(proba)
        r = np.random.rand()

        idx = np.searchsorted(cumsum, r)
        selected_value = index[idx]
        song_struct.append(selected_value)

        if selected_value == 6 :
            end = True

    print([states[i] for i in song_struct])
    return([states[i] for i in song_struct])

The char token representing the end of a section is at place 79

In [21]:
struct = generate_a_song_structure(prob_transi.astype(float),states)

encoded = torch.tensor([char2int[c] for c in struct[1]], dtype=torch.long).unsqueeze(0)

['<BEGINNING>', '<COUPLET>', '<PONT>', '<COUPLET>', '<REFRAIN>', '<COUPLET>', '<REFRAIN>', '<END>']


In [24]:
hid = model.init_hidden(batch_size=1)

for i in range(len(struct)-2) :

    print()
    print(struct[i+1])
    encoded = torch.tensor([char2int[c] for c in struct[i+1]], dtype=torch.long).unsqueeze(0)

    # prime the model with your encoded context
    out, hid = model(encoded, hid)

    # take the last token of the context
    next_input = encoded[:, -1].unsqueeze(0)  # shape (1, 1)

    last_input = 0

    while last_input != 79 :
        with torch.no_grad():
            out, hid = model(next_input, hid)   # out shape: (1, 1, vocab_size)

    # sample from logits
        next_token = sample_with_temp_topk(out[0, -1], temperature=0.4, top_k=40)
        last_input = next_token.item()

    # print char
        if next_token.item() != 79 :
            print(int2char[next_token.item()], end="")

    # prepare next input (still (1, 1))
        next_input = next_token.view(1, 1)

print("\n",struct[-1])


<COUPLET>

Moi je me le salaire
J'ai tout les forts de connaître ma source mais que la carrière de la prochain de la cours de la marque de la huile dans les voix de la paix d'un si je suis un milliers de ma dans les moments du passer la porte les portes s'aime et les rappeurs de l'Afrique de la marches de la grand-tu pas de mal d'une force de la puissance de la vie de la connutes de la leur tourne le fin d'un années au peuple pour les provoin de la couper les pommes de mon blanc et des mains de bien contre le sang
Algérien au trop de place de la meilleures conferme de l'amour de l'arme, le blancs sont le mot conscisseurs de la patrienne en corper, tout le regard de l'école et le rap et la proche de la manières de la malins
Mon rempli sur la rues de la proventes de la faute de la mains à transforme de la musique
C'est la grands pire de la vie de la couplets de conseils sont tout des doupe de nos coups de la paix de l'arme
En poussière de mon comptes de l'hommes de l'imposent
Mais pour 